Processing Categories to Apparatus as well as incidents

In [1]:
import geopandas as gpd
import pandas as pd
incidents_df= gpd.read_file("data/Vanderbilt_Fire/FireIncidents.geojson")
zones=gpd.read_file("data/beats_shpfile.geojson")
apparatus_df = pd.read_csv("data/Vanderbilt_Fire/Apparatus.csv")

NFDResponse = pd.read_csv("data/NFDResponse.csv")
cat_to_app = pd.read_csv("data/cat_to_apparatus.csv")



/var/folders/tf/7k8vj9w17715wv9yqsqn7pm00000gq/T/ipykernel_44152/1833893164.py:5: DtypeWarning: Columns (5,6,7,8,9,10,11,13,14,15,17,49,161,166) have mixed types. Specify dtype option on import or set low_memory=False.
  apparatus_df = pd.read_csv("data/Vanderbilt_Fire/Apparatus.csv")


In [2]:
apparatus_df

,Apparatus_ID_Internal,Incident_ID_Internal,Apparatus_Personnel_ID_List,Apparatus_Personnel_Name_List,Apparatus_Resource_Actions_Taken_1,Apparatus_Resource_Actions_Taken_2,Apparatus_Resource_Actions_Taken_3,Apparatus_Resource_Actions_Taken_4,Apparatus_Resource_Actions_Taken_Code_1,Apparatus_Resource_Actions_Taken_Code_2,...,Apparatus_Resource_Last_Arrived_At_Scene_Date_Time,Apparatus_Resource_Primary_Action_Taken,Apparatus_Resource_Primary_Action_Taken_Code,Apparatus_Resource_Primary_Action_Taken_Code_And_Description,Apparatus_Resource_Arrival_Sequence_Number_By_Overall_Incident,Apparatus_Resource_Arrival_Sequence_Number_By_Apparatus_Type,Apparatus_Resource_Narrative,Apparatus_Resource_Dispatch_Location,Apparatus_Resource_First_Unit_Arrived_To_Last_Unit_Arrived_in_Se,Apparatus_Resource_First_Unit_Arrived_To_Last_Unit_Arrived_in_Mi
0,3224082,1390914,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1.0,1.0,NaN,NaN,NaN,NaN
1,3224086,1390917,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1.0,1.0,NaN,NaN,NaN,NaN
2,3224087,1390917,"716801, 239398, 434994, 824096","Henry (Dillon) Brackman, David Christian, Cody...",NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,Cancelled en route,93,93 - Cancelled en route,NaN,NaN,RE12C was cancelled en route by MED15 and retu...,NaN,NaN,NaN
3,3224088,1390918,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2.0,2.0,NaN,NaN,NaN,NaN
4,3224091,1390920,"224513, 4001356, 461776","Julie Haynes, Andrea Mason, Constance Swett",NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,Provide basic life support (BLS),32,32 - Provide basic life support (BLS),1.0,1.0,EN25C see medic report. EN25C back in service.,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1381986,7577209,3190793,"416193, 309272, 4006534","Lawrence Jr Williams, Jacob Welbaum, Austin (C...",NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,Provide basic life support (BLS),32.0,32 - Provide basic life support (BLS),1.0,1.0,See med report,NaN,NaN,NaN
1381987,7577210,3190794,"438458, 269278, 4007124","David Monast, Steven (Michael) Ward, Devan Pierce",NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1.0,1.0,NaN,NaN,NaN,NaN
1381988,7577211,3190794,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2.0,1.0,NaN,NaN,NaN,NaN
1381989,7577212,3190795,"161923, 4010901, 4011604","John Davis, Levi Jr. Johnson, Clayton Yates",NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,Provide manpower,73.0,73 - Provide manpower,1.0,1.0,see medical report,NaN,NaN,NaN


In [3]:
apparatus_df[apparatus_df.Incident_ID_Internal==3190794]['Apparatus_Resource_Arrived_To_Cleared_In_Minutes']

1381987     4.73
1381988    65.12
Name: Apparatus_Resource_Arrived_To_Cleared_In_Minutes, dtype: float64

In [4]:
def combine_zones(gdf: gpd.GeoDataFrame, zone_id_col: str = "ZONE_ID") -> gpd.GeoDataFrame:
    """Combine polygons with the same zone ID into single polygons."""
    df_copy = gdf.copy()
    df_copy = df_copy.dissolve(by=zone_id_col,).reset_index()
    return df_copy
zones = combine_zones(zones, zone_id_col="ZONE_ID",)

In [5]:
apparatus_df.Incident_ID_Internal = apparatus_df.Incident_ID_Internal.astype(int)
apparatus_df=apparatus_df[apparatus_df.Incident_ID_Internal.isin(incidents_df['IncidentIDInternal'].astype(int))]
app_to_stations=pd.read_csv('data/ApparatusID_to_Station.csv')
app_to_stations.rename(columns={'Station': 'Facility Name'}, inplace=True)
app_to_stations.loc[app_to_stations['Facility Name'].str.contains('Goodlettsville Fire', na=False), 'Facility Name'] = 41
app_to_stations = app_to_stations[pd.to_numeric(app_to_stations['Facility Name'], errors='coerce').notna()]
app_to_stations['Facility Name'] = app_to_stations['Facility Name'].apply(lambda x: f"Station {int(x):02d}")


In [6]:
#split rows with "/" in Apparatus_Resource_ID
app_to_stations['Apparatus_Resource_ID'] = app_to_stations['Apparatus_Resource_ID'].str.split('/')
app_to_stations = app_to_stations.explode('Apparatus_Resource_ID')


In [7]:
#DROP ALL COLUMNS THAT HAVE MORE THAN 90% MISSING VALUES IN APPARATUS_DF
threshold = len(apparatus_df) * 0.1
apparatus_df = apparatus_df.dropna(thresh=threshold, axis=1)

In [8]:
apparatus=pd.merge(apparatus_df, app_to_stations, on='Apparatus_Resource_ID', how='left')
apparatus.sort_values(['Incident_ID_Internal','Apparatus_Resource_Arrival_Sequence_Number_By_Apparatus_Type'], inplace=True)


In [9]:
apparatus=apparatus[apparatus['Facility Name'].notna()].groupby('Incident_ID_Internal').first().reset_index()

In [10]:
incidents_df['IncidentIDInternal']=incidents_df['IncidentIDInternal'].astype(int)
incidents_df=pd.merge(incidents_df,apparatus, left_on='IncidentIDInternal',right_on='Incident_ID_Internal')
# incidents_df[incidents_df['IncidentIDInternal'].astype(int).isin(apparatus['Incident_ID_Internal'])]


In [11]:
incidents_df

,IncidentNumber,NFIRSCode,NFIRSType,NFPACategoryCode,NFPACategory,IncidentModifiedDate,FireModifiedDate,PSAPDate,AlarmDate,ArrivalDate,...,Apparatus_Resource_Vehicle_Call_Sign,Apparatus_Resource_Vehicle_Category_Type,Apparatus_Resource_Primary_Action_Taken,Apparatus_Resource_Primary_Action_Taken_Code,Apparatus_Resource_Primary_Action_Taken_Code_And_Description,Apparatus_Resource_Arrival_Sequence_Number_By_Overall_Incident,Apparatus_Resource_Arrival_Sequence_Number_By_Apparatus_Type,Apparatus_Resource_Narrative,Facility Name,Apparatus_Type
0,FFD250101000003,700,False Alarm False Call,25,Non-Fire Incidents,2025-02-06 16:03:36+00:00,2025-02-06 16:19:36+00:00,2025-01-01 00:04:00+00:00,2025-01-01 00:06:02+00:00,2025-01-01 00:11:16+00:00,...,EN24,"Engine, Fire (Pumper) - Type I",Investigate,86,86 - Investigate,1.0,1.0,EN24 arrived on the scene to two story gable r...,Station 24,Engine
1,FFD250101000004,300,EMS & Rescue,25,Non-Fire Incidents,2025-02-06 16:03:36+00:00,2025-02-06 16:19:36+00:00,2025-01-01 00:06:45+00:00,2025-01-01 00:07:41+00:00,2025-01-01 00:15:46+00:00,...,MED02,Fire Truck - Aerial (Ladder or Platform) - Type I,Provide manpower,73,73 - Provide manpower,2.0,1.0,TW02 A arrived on scene along with medic unit....,Station 09,Medic
2,FFD250101000005,100,Fire,19,Non-Structure Fire,2025-02-06 16:03:36+00:00,2025-02-06 16:19:36+00:00,2025-01-01 00:07:50+00:00,2025-01-01 00:08:06+00:00,2025-01-01 00:14:14+00:00,...,EN35,"Engine, Fire (Pumper) - Type I","Fire control or extinguishment, other",10,"10 - Fire control or extinguishment, other",1.0,1.0,ENG-35-A\n we responded to report of car fire ...,Station 35,Engine
3,FFD250101000006,600,Good Intent Call,25,Non-Fire Incidents,2025-02-06 16:03:36+00:00,2025-02-06 16:19:36+00:00,2025-01-01 00:06:36+00:00,2025-01-01 00:10:05+00:00,NaT,...,DE05,"Engine, Fire (Pumper) - Type I",Cancelled en route,93,93 - Cancelled en route,NaN,NaN,EN21A- Staged and Cancelled.\nRTS.,Station 05,District EMS
4,FFD250101000009,300,EMS & Rescue,25,Non-Fire Incidents,2025-02-06 16:03:36+00:00,2025-02-06 16:19:36+00:00,2025-01-01 00:10:06+00:00,2025-01-01 00:11:32+00:00,2025-01-01 00:16:37+00:00,...,MED05,None,None,None,None,1.0,1.0,None,Station 05,Medic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
544640,FFD250722100602,300,EMS & Rescue,None,None,2025-07-23 07:24:03+00:00,2025-07-23 07:24:42+00:00,2025-07-22 23:49:42+00:00,2025-07-22 23:50:01+00:00,2025-07-23 00:08:48+00:00,...,MED03,None,None,None,None,1.0,1.0,None,Station 03,Medic
544641,FFD250722100603,100,Fire,19,Non-Structure Fire,2025-07-23 10:44:44+00:00,2025-07-23 10:45:19+00:00,2025-07-22 23:50:20+00:00,2025-07-22 23:51:55+00:00,2025-07-23 00:03:08+00:00,...,EN03,"Engine, Fire (Pumper) - Type I",Investigate,86.0,86 - Investigate,1.0,1.0,eng 3 on scene for a man that had side swiped ...,Station 03,Engine
544642,FFD250722100605,300,EMS & Rescue,25,Non-Fire Incidents,2025-07-23 06:07:34+00:00,2025-07-23 06:08:10+00:00,2025-07-22 23:53:37+00:00,2025-07-22 23:54:14+00:00,2025-07-22 23:57:52+00:00,...,EN31,"Engine, Fire (Pumper) - Type I",Standby (Staged at incident),92.0,92 - Standby (Staged at incident),1.0,1.0,Engine 31 arrived and was canceled immediately...,Station 31,Engine
544643,FFD250722100604,300,EMS & Rescue,25,Non-Fire Incidents,2025-07-23 05:50:02+00:00,2025-07-23 05:50:37+00:00,2025-07-22 23:52:36+00:00,2025-07-22 23:52:53+00:00,2025-07-22 23:58:32+00:00,...,EN06,"Engine, Fire (Pumper) - Type I","Assistance, other",70.0,"70 - Assistance, other",2.0,1.0,EN06C arrived on scene with MED06 to an indivi...,Station 06,Engine


In [12]:

type_cats=['EMSApparatusCount', 'OtherApparatusCount',
       'SuppressionApparatusCount', 'OtherPersonnelCount','PrimaryActionTaken']
incidents_export=pd.merge(incidents_df,pd.merge(cat_to_app,NFDResponse, left_on= "MappedCategory", right_on="Category", how="inner"), on='IncidentType', how='left')

In [13]:
incidents_export

,IncidentNumber,NFIRSCode,NFIRSType,NFPACategoryCode,NFPACategory,IncidentModifiedDate,FireModifiedDate,PSAPDate,AlarmDate,ArrivalDate,...,Rescue,Hazard,Squad,FAST,Medic,Brush,Boat,UTV,REACH,Chief
0,FFD250101000003,700,False Alarm False Call,25,Non-Fire Incidents,2025-02-06 16:03:36+00:00,2025-02-06 16:19:36+00:00,2025-01-01 00:04:00+00:00,2025-01-01 00:06:02+00:00,2025-01-01 00:11:16+00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
1,FFD250101000004,300,EMS & Rescue,25,Non-Fire Incidents,2025-02-06 16:03:36+00:00,2025-02-06 16:19:36+00:00,2025-01-01 00:06:45+00:00,2025-01-01 00:07:41+00:00,2025-01-01 00:15:46+00:00,...,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN
2,FFD250101000005,100,Fire,19,Non-Structure Fire,2025-02-06 16:03:36+00:00,2025-02-06 16:19:36+00:00,2025-01-01 00:07:50+00:00,2025-01-01 00:08:06+00:00,2025-01-01 00:14:14+00:00,...,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,FFD250101000006,600,Good Intent Call,25,Non-Fire Incidents,2025-02-06 16:03:36+00:00,2025-02-06 16:19:36+00:00,2025-01-01 00:06:36+00:00,2025-01-01 00:10:05+00:00,NaT,...,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN
4,FFD250101000009,300,EMS & Rescue,25,Non-Fire Incidents,2025-02-06 16:03:36+00:00,2025-02-06 16:19:36+00:00,2025-01-01 00:10:06+00:00,2025-01-01 00:11:32+00:00,2025-01-01 00:16:37+00:00,...,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
544640,FFD250722100602,300,EMS & Rescue,None,None,2025-07-23 07:24:03+00:00,2025-07-23 07:24:42+00:00,2025-07-22 23:49:42+00:00,2025-07-22 23:50:01+00:00,2025-07-23 00:08:48+00:00,...,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN
544641,FFD250722100603,100,Fire,19,Non-Structure Fire,2025-07-23 10:44:44+00:00,2025-07-23 10:45:19+00:00,2025-07-22 23:50:20+00:00,2025-07-22 23:51:55+00:00,2025-07-23 00:03:08+00:00,...,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
544642,FFD250722100605,300,EMS & Rescue,25,Non-Fire Incidents,2025-07-23 06:07:34+00:00,2025-07-23 06:08:10+00:00,2025-07-22 23:53:37+00:00,2025-07-22 23:54:14+00:00,2025-07-22 23:57:52+00:00,...,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN
544643,FFD250722100604,300,EMS & Rescue,25,Non-Fire Incidents,2025-07-23 05:50:02+00:00,2025-07-23 05:50:37+00:00,2025-07-22 23:52:36+00:00,2025-07-22 23:52:53+00:00,2025-07-22 23:58:32+00:00,...,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN


In [14]:
incidents_export['test_time'] = incidents_export['AlarmLastUnitClearTime'] - incidents_export['FirstEngineArrivalTime']
incidents_export['test_time'].fillna( incidents_export['AlarmLastUnitClearTime'] - incidents_export['FirstEMSArrivalTime'],inplace=True)

/var/folders/tf/7k8vj9w17715wv9yqsqn7pm00000gq/T/ipykernel_44152/2330962784.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  incidents_export['test_time'].fillna( incidents_export['AlarmLastUnitClearTime'] - incidents_export['FirstEMSArrivalTime'],inplace=True)


In [15]:
incidents_export

,IncidentNumber,NFIRSCode,NFIRSType,NFPACategoryCode,NFPACategory,IncidentModifiedDate,FireModifiedDate,PSAPDate,AlarmDate,ArrivalDate,...,Hazard,Squad,FAST,Medic,Brush,Boat,UTV,REACH,Chief,test_time
0,FFD250101000003,700,False Alarm False Call,25,Non-Fire Incidents,2025-02-06 16:03:36+00:00,2025-02-06 16:19:36+00:00,2025-01-01 00:04:00+00:00,2025-01-01 00:06:02+00:00,2025-01-01 00:11:16+00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,186.0
1,FFD250101000004,300,EMS & Rescue,25,Non-Fire Incidents,2025-02-06 16:03:36+00:00,2025-02-06 16:19:36+00:00,2025-01-01 00:06:45+00:00,2025-01-01 00:07:41+00:00,2025-01-01 00:15:46+00:00,...,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,3193.0
2,FFD250101000005,100,Fire,19,Non-Structure Fire,2025-02-06 16:03:36+00:00,2025-02-06 16:19:36+00:00,2025-01-01 00:07:50+00:00,2025-01-01 00:08:06+00:00,2025-01-01 00:14:14+00:00,...,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2767.0
3,FFD250101000006,600,Good Intent Call,25,Non-Fire Incidents,2025-02-06 16:03:36+00:00,2025-02-06 16:19:36+00:00,2025-01-01 00:06:36+00:00,2025-01-01 00:10:05+00:00,NaT,...,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN
4,FFD250101000009,300,EMS & Rescue,25,Non-Fire Incidents,2025-02-06 16:03:36+00:00,2025-02-06 16:19:36+00:00,2025-01-01 00:10:06+00:00,2025-01-01 00:11:32+00:00,2025-01-01 00:16:37+00:00,...,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,3117.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
544640,FFD250722100602,300,EMS & Rescue,None,None,2025-07-23 07:24:03+00:00,2025-07-23 07:24:42+00:00,2025-07-22 23:49:42+00:00,2025-07-22 23:50:01+00:00,2025-07-23 00:08:48+00:00,...,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,3563.0
544641,FFD250722100603,100,Fire,19,Non-Structure Fire,2025-07-23 10:44:44+00:00,2025-07-23 10:45:19+00:00,2025-07-22 23:50:20+00:00,2025-07-22 23:51:55+00:00,2025-07-23 00:03:08+00:00,...,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,630.0
544642,FFD250722100605,300,EMS & Rescue,25,Non-Fire Incidents,2025-07-23 06:07:34+00:00,2025-07-23 06:08:10+00:00,2025-07-22 23:53:37+00:00,2025-07-22 23:54:14+00:00,2025-07-22 23:57:52+00:00,...,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,2901.0
544643,FFD250722100604,300,EMS & Rescue,25,Non-Fire Incidents,2025-07-23 05:50:02+00:00,2025-07-23 05:50:37+00:00,2025-07-22 23:52:36+00:00,2025-07-22 23:52:53+00:00,2025-07-22 23:58:32+00:00,...,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,1361.0


In [16]:
incidents_export['response_time'] = incidents_export['AlarmLastUnitClearTime'] - incidents_export['AlarmFirstUnitArriveTime']
incidents_export['resolution_time'] = (incidents_export['LastUnitClearedDate'] - incidents_export['PSAPDate']).dt.total_seconds()
incidents_export = incidents_export[( incidents_export['SuppressionPersonnelCount'].notna())&(incidents_export['EMSApparatusCount'].notna())& ((incidents_export['response_time'].notna())&(incidents_export['response_time']>= 0)) &(incidents_export['AlarmFirstUnitArriveTime']>0) & (incidents_export['AlarmLastUnitClearTime']>0)]
grouped_incidents = incidents_export.groupby('Enum')


for name, group in grouped_incidents:
    q99 = group['response_time'].quantile(0.99)
    q98= group['AlarmFirstUnitArriveTime'].quantile(0.99)
    incidents_export = incidents_export[~((incidents_export['Enum'] == name) & (incidents_export['response_time'] > q99))]
    incidents_export = incidents_export[~((incidents_export['Enum'] == name) & (incidents_export['AlarmFirstUnitArriveTime'] > q98))]

In [17]:
metrics = ["response_time"]
metrics_desc = []

for metric in metrics:
    incidents_export[metric] = incidents_export[metric].astype(float)  # Ensure numeric type
    summary = (
        incidents_export
          .groupby("Enum")[metric]
          .describe()
    )
    summary.Name = metric
    summary = summary.reset_index()
    summary.to_csv(f"data/exploratory_analysis/{metric}_summary2.csv", index=False)
    # print(f"Summary for {metric} saved to data/exploratory_analysis/{metric}_summary.csv")incidents_export

In [18]:
action_severity_map = {
    "Investigate": "Low",
    "Provide manpower": "Low",
    "Extinguishment by fire service personnel": "High",
    "Search & rescue, other": "High",
    "Rescue, remove from harm": "High",
    "Provide basic life support (BLS)": "Moderate",
    "Emergency medical services, other": "Moderate",
    "Provide first aid & check for injuries": "Moderate",
    "Assistance, other": "Low",
    "Standby  (Staged on Scene)": "Low",
    "Assist physically disabled": "Moderate",
    "Hazardous materials spill control and confinement": "High",
    "Provide advanced life support (ALS)": "High",
    "Investigate fire out on arrival": "Low",
    "HazMat detection, monitoring, sampling, & analysis": "High",
    "Forcible entry": "Moderate",
    "Extricate, disentangle": "High",
    "Information, investigation & enforcement, other": "Low",
    "Shut down system": "Low",
    "Incident command": "Critical",
    "Assist animal": "Low",
    "Control traffic": "Low",
    "Secure property": "Low",
    "Ventilate B (Horizontal Ventilation)": "Moderate",
    "Notify other agencies.": "Low",
    "Remove hazard": "Moderate",
    "Refer to proper authority": "Low",
    "Provide information to public or media": "Low",
    "Restore fire alarm system": "Low",
    "Search": "Moderate",
    "Establish safe area": "Moderate",
    "Ventilate C (Smoke Removal Only)": "Moderate",
    "Action taken, other": "Low",
    "Provide apparatus": "Moderate",
    "Evacuate area": "High",
    "Remove water": "Low",
    "Identify, analyze hazardous materials": "High",
    "Provide equipment": "Moderate",
    "Hazardous materials leak control & containment": "High",
    "Enforce codes": "Low",
    "Restore sprinkler or fire protection system": "Low",
    "Fires, rescues & hazardous conditions, other": "High",
    "Remove hazardous materials": "High",
    "Transport person": "Moderate",
    "Systems and services, other": "Low",
    "Decontaminate persons or equipment": "High",
    "Recover body": "High",
    "Determine if materials are non-hazardous": "Low",
    "Provide water": "Moderate",
    "Ventilate A (Vertical Ventilation)": "Moderate",
    "Control fire (wildland)": "Critical",
    "Salvage & overhaul": "Moderate",
    "Hazardous condition, other": "Moderate",
    "Assess severe weather or natural disaster damage": "High",
    "Fill-in or moveup  (Back up)": "Low",
    "Operate apparatus or vehicle": "Low",
    "Decontaminate occupancy or area": "High",
    "Provide air supply": "Low",
    "Restore municipal services": "Moderate",
    "Contain fire (wildland)": "Critical",
    "Manage prescibed fire (wildland)": "High",
    "Provide light or electrical power": "Low",
    "Cancelled en route": "Low",
    "Establish fire lines (wildfire)": "Critical",
    "Confine fire (wildland)": "Critical",
    "Control crowd": "Low",
    "Fire control or extinguishment, other": "High"
}


In [19]:
incidents_export['incident_level'] = incidents_export['PrimaryActionTaken'].map(action_severity_map)
low = (incidents_export['incident_level'].isnull()) & (incidents_export['EMSApparatusCount'] >= 0) & (incidents_export['EMSApparatusCount'] <= 2) & (incidents_export['SuppressionApparatusCount'] == 0) 
moderate = (incidents_export['incident_level'].isnull()) & ((incidents_export['EMSApparatusCount'] > 2) | ((incidents_export['SuppressionApparatusCount'] >= 0) & (incidents_export['SuppressionApparatusCount'] <= 2)))
high = (incidents_export['incident_level'].isnull()) & (incidents_export['SuppressionApparatusCount'] > 2) & (incidents_export['SuppressionApparatusCount']<= 5) 
critical = (incidents_export['incident_level'].isnull()) & (incidents_export['SuppressionApparatusCount'] > 5) 
incidents_export.loc[low, 'incident_level'] = 'Low'
incidents_export.loc[moderate, 'incident_level'] = 'Moderate'
incidents_export.loc[high, 'incident_level'] = 'High'
incidents_export.loc[critical, 'incident_level'] = 'Critical'


In [20]:

# rename_cats=['Latitude','Longitude','NFIRSType','incident','PSAPDate','Enum']
cats=['incident_id','lat','lon','incident_type','incident_level','datetime','category']
incidents_export.rename(columns={'Longitude': 'lon', 'Latitude': 'lat','IncidentType': 'incident_type','PSAPDate': 'datetime','Enum': 'category'}, inplace=True)
incidents_export = incidents_export.dropna(subset=['lat', 'lon', 'incident_type', 'incident_level', 'datetime','Facility Name'])
incidents_export.sort_values('datetime',inplace=True)
incidents_export.reset_index(drop=True, inplace=True)
incidents_export.reset_index(names='incident_id', inplace=True)

In [21]:
incidents_export['datetime']=incidents_export['datetime'].dt.strftime('%Y-%m-%d %H:%M:%S')
incidents_export['category'] = incidents_export['category'].str.strip()
incidents_export['incident_type'] = incidents_export['incident_type'].replace({',':' - ' }, regex=True)



In [22]:
incidents_export=incidents_export.merge(app_to_stations[['Facility Name','Apparatus_Resource_ID']],left_on='FirstEngine',right_on='Apparatus_Resource_ID',suffixes=('','_ENG'),how='left')
incidents_export=incidents_export.merge(app_to_stations[['Facility Name','Apparatus_Resource_ID']],left_on='FirstEMS',right_on='Apparatus_Resource_ID',suffixes=('','_EMS'),how='left')

In [23]:
fire_condition=False

In [24]:

if not fire_condition:
    incidents_export[cats].to_csv("data/incidents_export_apparatus.csv", index=False)
else:
    incidents_export[incidents_export.NFIRSCode=='100'][cats].to_csv("data/incidents_export_apparatus_fire.csv", index=False)

In [25]:

# incident_resolution_df = incidents_export[['incident_id', 'AlarmArriveTime','response_time', 'resolution_time','Facility Name']].copy()
if fire_condition:
    incident_resolution_df = incidents_export[incidents_export.NFIRSCode=='100'][['incident_id', 'AlarmArriveTime','response_time', 'resolution_time','Facility Name','FirstEngine','FirstEMS','FirstEngineTravelTime','FirstEngineArrivalDate','FirstEMS','FirstEMSArrivalDate','FirstEMSTravelTime','Facility Name_EMS','Facility Name_ENG','Apparatus_Resource_Arrived_To_Cleared_In_Minutes']].copy()
    incident_resolution_df.to_csv("data/incident_resolution_times_fire.csv", index=False)
else:
    incident_resolution_df = incidents_export[['incident_id', 'AlarmArriveTime','response_time', 'resolution_time','Facility Name','FirstEngine','FirstEMS','FirstEngineTravelTime','FirstEngineArrivalDate','FirstEMS','FirstEMSArrivalDate','FirstEMSTravelTime','Facility Name_EMS','Facility Name_ENG','Apparatus_Resource_Arrived_To_Cleared_In_Minutes']].copy()
    incident_resolution_df.to_csv("data/incident_resolution_times.csv", index=False)

In [ ]:
org_dest=incidents_export[['IncidentNumber','lat','lon','Facility Name']]
org_dest.rename(columns={'lat':'dest_lat','lon':'dest_lon'}, inplace=True)
stations=pd.read_csv("data/stations.csv")
org_dest=org_dest.merge(stations[['Facility Name','lat','lon']], on='Facility Name', how='left')
org_dest.rename(columns={'lat':'org_lat','lon':'org_lon'}, inplace=True)
org_dest=org_dest[['IncidentNumber','org_lat','org_lon','dest_lat','dest_lon',]]


In [ ]:
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import time

def get_travel_time(origin_lon, origin_lat, dest_lon, dest_lat, osrm_url="http://localhost:8080"):
    """Get travel time in seconds using OSRM container"""
    try:
        url = f"{osrm_url}/route/v1/driving/{origin_lon},{origin_lat};{dest_lon},{dest_lat}"
        response = requests.get(url, params={'overview': 'false'}, timeout=5)
        
        if response.status_code == 200:
            data = response.json()
            if data['code'] == 'Ok':
                return data['routes'][0]['duration']
        return None
    except:
        return None

def add_travel_times_fast(df, origin_lon_col, origin_lat_col, dest_lon_col, dest_lat_col, max_workers=20):
    """Add travel time column to dataframe using parallel processing"""
    def get_time_for_row(args):
        idx, row = args
        time.sleep(0.01)  # Small delay to avoid overwhelming server
        travel_time = get_travel_time(row[origin_lon_col], row[origin_lat_col], 
                                    row[dest_lon_col], row[dest_lat_col])
        return idx, travel_time
    
    # Create list of (index, row) tuples
    row_data = [(idx, row) for idx, row in df.iterrows()]
    
    # Process in parallel
    travel_times = [None] * len(df)
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks
        futures = [executor.submit(get_time_for_row, row_item) for row_item in row_data]
        
        # Collect results with progress bar
        for future in tqdm(as_completed(futures), total=len(futures), desc="Calculating travel times"):
            try:
                idx, travel_time = future.result()
                travel_times[idx] = travel_time
            except Exception as e:
                print(f"Error processing row: {e}")
    
    # Add results to dataframe
    df_copy = df.copy()
    df_copy['travel_time_seconds'] = travel_times
    df_copy['travel_time_minutes'] = [t/60 if t is not None else None for t in travel_times]
    
    # Print stats
    successful = sum(1 for t in travel_times if t is not None)
    print(f"Success rate: {successful}/{len(travel_times)} ({successful/len(travel_times)*100:.1f}%)")
    
    return df_copy

# df_with_times = add_travel_times_fast(org_dest, 'org_lon', 'org_lat', 'dest_lon', 'dest_lat')

# df_with_times = df_with_times.merge(incidents_export[['IncidentNumber','datetime','AlarmArriveTime','Facility Name']], left_on='IncidentNumber', right_on='IncidentNumber', how='left')
# df_with_times.rename(columns={'travel_time_seconds': 'osrm_travel_time_seconds', 'travel_time_minutes': 'osrm_travel_time_minutes'}, inplace=True)
# df_with_times.to_csv("data/incidents_with_travel_times_OSRM.csv", index=False)

In [ ]:
import numpy as np
#get the metrics summary of bias, variance, rmse, mae, mape
def metrics_summary(df: pd.DataFrame,
                    actual_col: str = "Alarm_Time",
                    pred_col: str = "travel_time_seconds") -> pd.Series:
    """
    Returns bias, variance (of error), RMSE, MAE, MAPE as a pandas Series.
    - bias = mean(pred - actual)
    - variance = var(pred - actual), ddof=0
    - RMSE = sqrt(mean((pred - actual)^2))
    - MAE = mean(|pred - actual|)
    - MAPE = mean(|(pred - actual)/actual|)*100, computed only where actual>0
    """
    y_true = df[actual_col]
    y_pred = df[pred_col]

    mask = y_true.notna() & y_pred.notna()
    y_true = y_true[mask].to_numpy(dtype=float)
    y_pred = y_pred[mask].to_numpy(dtype=float)

    err = y_pred - y_true

    bias = np.nanmean(err)
    var_err = np.nanvar(err)  # population variance
    rmse = np.sqrt(np.nanmean(err**2))
    mae = np.nanmean(np.abs(err))

    # MAPE on positive actuals to avoid div-by-zero
    pos = y_true > 0
    mape = np.nan if not np.any(pos) else np.nanmean(np.abs(err[pos] / y_true[pos])) * 100.0

    return pd.Series({
        "bias_seconds": bias,
        "variance_seconds2": var_err,
        "rmse_seconds": rmse,
        "mae_seconds": mae,
        "mape_percent": mape
    })

# summary = metrics_summary(df_with_times, actual_col="AlarmArriveTime", pred_col="travel_time_seconds")


In [26]:
features=['incident_id',
 'incident_type',
 'datetime',
 'lat',
 'lon',
'category',
 'response_time',
 'resolution_time',
 'Facility Name',
 'AlarmArriveTime',
 'NFIRSType'
 ]
apparatus_features=['Engine_ID', 'Truck', 'Rescue', 'Hazard', 'Squad', 'FAST', 'Medic',
       'Brush', 'Boat', 'UTV', 'REACH', 'Chief']
incidents_fire=incidents_export[incidents_export.NFIRSCode=='100']
incidents=gpd.GeoDataFrame(incidents_fire, geometry=gpd.points_from_xy(incidents_fire.lon, incidents_fire.lat), crs="EPSG:4326")
# incidents=incidents[features + ['geometry']+apparatus_features]
incidents=gpd.sjoin(incidents, zones, how="left", predicate='within').drop(columns=['index_right'])

incidents.dropna(subset=['ZONE_ID'], inplace=True)

incidents.drop(columns=['geometry','ZONE','TYPE','NAME','Facility Name','AlarmArriveTime','NFIRSType']+apparatus_features).to_csv("data/incidents_data_for_modeling.csv", index=False)

In [ ]:
stations= gpd.read_file("data/stations.csv")
stations=gpd.GeoDataFrame(stations, geometry=gpd.points_from_xy(stations.lon, stations.lat), crs="EPSG:4326")
stations=stations[['Facility Name', 'lat', 'lon', 'geometry']]
stations=stations.sjoin( zones[['ZONE_ID','geometry']], how="left", predicate='within').drop(columns=['index_right'])


In [ ]:
incidents=incidents.merge(stations[['Facility Name', 'lat', 'lon','ZONE_ID']], left_on='Facility Name', right_on='Facility Name', how='left',suffixes=('', '_station'))

In [ ]:

travel_time_features=['incident_id','incident_type','category','NFIRSType','datetime','lat','lon','ZONE_ID','Facility Name','lat_station','lon_station','ZONE_ID_station','AlarmArriveTime'] +apparatus_features
incidents[travel_time_features].to_csv("data/incidents_travel_time_features.csv", index=False)

In [ ]:

incidents['StationName']=incidents['Facility Name']
zone_groups=incidents.groupby('ZONE_ID')


In [ ]:
zones_list=zones.ZONE_ID.unique().tolist()

In [ ]:
import numpy as np
zone_to_zone=incidents.pivot_table(index='ZONE_ID_station', columns='ZONE_ID', values='FirstEngineTravelTime', aggfunc='mean')
unreachable_zones= list(set(zones_list) - set(zone_to_zone.columns))
zone_to_zone[unreachable_zones]=np.nan
zone_to_zone=zone_to_zone[zone_to_zone.columns.sort_values()]


In [ ]:
zone_to_zone_std=incidents.pivot_table(index='ZONE_ID_station', columns='ZONE_ID', values='FirstEngineTravelTime', aggfunc='std',fill_value=0)
unreachable_zones= list(set(zones_list) - set(zone_to_zone_std.columns))
zone_to_zone_std[unreachable_zones]=0.00
zone_to_zone_std=zone_to_zone_std[zone_to_zone_std.columns.sort_values()]

In [ ]:
zone_to_zone_std

In [ ]:
zone_fire_station_info=incidents.groupby(by=['ZONE_ID_station','lat_station','lon_station']).size().reset_index()
zone_fire_station_info.rename(columns={"lat_station":'lat',"lon_station":'lon'},inplace=True)
zone_fire_station_info.drop(0,axis=1,inplace=True)

In [ ]:
import os
os.makedirs("data/interpolation_fire", exist_ok=True)
zone_to_zone_std.columns = zone_to_zone_std.columns.astype(int)
import json
with open("data/interpolation_fire/std_zone_travel_time_matrix.json", "w") as f:
    json.dump(zone_to_zone_std.to_dict(orient='index'), f,indent=4)
with open("data/interpolation_fire/zone_fire_station_info.json", "w") as f:
    json.dump(zone_fire_station_info.set_index('ZONE_ID_station').to_dict(orient='index'), f,indent=4)


In [ ]:

#fill out missing values with weighted alarm times of nearby zones multiplied by haversine ratio
haversine_distances = {}
for from_zone in zone_to_zone.columns:
    from_zone_data = incidents[incidents['ZONE_ID_station'] == from_zone]
    if from_zone_data.empty:
        zones_data = zones[zones['ZONE_ID'] == from_zone]
        from_lat = float(zones_data['geometry'].iloc[0].centroid.y)
        from_lon = float(zones_data['geometry'].iloc[0].centroid.x)
    else:
        from_lat = float(from_zone_data['lat_station'].iloc[0])
        from_lon = float(from_zone_data['lon_station'].iloc[0])

    haversine_distances[from_zone] = {}
    
    for to_zone in zone_to_zone.columns:
        to_zone_data = incidents[incidents['ZONE_ID'] == to_zone]
        if to_zone_data.empty:
            to_lat=float(zones[zones['ZONE_ID'] == to_zone]['geometry'].iloc[0].centroid.y)
            to_lon=float(zones[zones['ZONE_ID'] == to_zone]['geometry'].iloc[0].centroid.x)
        else:

            to_lat = float(to_zone_data['lat'].mean())
            to_lon = float(to_zone_data['lon'].mean())

        # Calculate haversine distance
        R = 6371.0  # Earth radius in kilometers
        dlat = np.radians(to_lat - from_lat)
        dlon = np.radians(to_lon - from_lon)
        a = (np.sin(dlat / 2) ** 2 +
             np.cos(np.radians(from_lat)) * np.cos(np.radians(to_lat)) * np.sin(dlon / 2) ** 2)
        c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
        distance = R * c  # in kilometers

        haversine_distances[from_zone][to_zone] = float(distance)

                


In [ ]:
zone_to_zone_counts=incidents.pivot_table(index='ZONE_ID_station', columns='ZONE_ID', values='incident_id', aggfunc='count',fill_value=0)

In [ ]:
zone_to_zone_filled = zone_to_zone.copy()
for from_zone in zone_to_zone.index:
    for to_zone in zone_to_zone.columns:
        if pd.isna(zone_to_zone.at[from_zone, to_zone]):
            # Get distances to all other zones with data
            distances = []
            weights = []
            weighted_values = []
            distance_from_to = haversine_distances[from_zone].get(to_zone, np.inf)
            for other_zone in zone_to_zone.columns:
                if other_zone != to_zone and not pd.isna(zone_to_zone.at[from_zone, other_zone]):
                    dist = haversine_distances[from_zone].get(other_zone, np.inf)
                    if dist > 0:
                        distances.append(dist)
                        weight =(1/haversine_distances[to_zone].get(other_zone, np.inf))**2
                        weights.append(weight)
                        weighted_values.append(zone_to_zone.at[from_zone, other_zone]* (distance_from_to / dist)* weight)

            if weights:
                weighted_avg = sum(weighted_values) / sum(weights)
                zone_to_zone_filled.at[from_zone, to_zone] = weighted_avg

In [ ]:
zone_to_zone_filled.columns = zone_to_zone_filled.columns.astype(int)
with open("data/interpolation_fire/mean_zone_travel_time_matrix.json", "w") as f:
    json.dump(zone_to_zone_filled.to_dict(orient='index'), f,indent=4)